In [ ]:
# Colab setup: clone the repo if needed, then install the extracted package.
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/haydenyoungcs/gradient-ascent.git"
REPO_DIR = pathlib.Path("/content/gradient-ascent")

github_token = os.environ.get("GITHUB_TOKEN")
wandb_api_key = os.environ.get("WANDB_API_KEY")

try:
    from google.colab import userdata  # type: ignore

    if github_token is None:
        github_token = userdata.get("GITHUB_TOKEN")
    if wandb_api_key is None:
        wandb_api_key = userdata.get("WANDB_API_KEY")
except Exception:
    pass

cwd = pathlib.Path.cwd()
project_root = cwd if (cwd / "pyproject.toml").exists() else REPO_DIR

if not project_root.exists():
    if github_token:
        clone_url = REPO_URL.replace("https://", f"https://{github_token}@")
        subprocess.run(["git", "clone", clone_url], check=True)
    else:
        raise RuntimeError(
            "Repo checkout not found. For this private repo, add a Colab secret or env var named "
            "GITHUB_TOKEN, or clone the repo manually before running this notebook."
        )

if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError(f"Expected pyproject.toml under {project_root}, but it was not found.")

os.chdir(project_root)
print(f"Changed working directory to {project_root}")

repo_src = project_root / "src"
for path in [project_root, repo_src]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(project_root), "wandb", "pot", "scikit-learn"], check=True)

import wandb

if wandb_api_key:
    wandb.login(key=wandb_api_key, relogin=True)
else:
    print("wandb installed; set WANDB_API_KEY if you want online logging.")


## 1-4. Setup, training, and core model checkpoints

This section trains the original and retrained models, then runs five unlearning baselines: gradient ascent, SSD, SalUn, certified removal, and SCRUB. For GA, the preset below stays faithful to vanilla gradient ascent on the forget set, but is made slightly stronger in an easy-to-justify way: a modestly higher learning rate, more forget-set updates per epoch, BatchNorm buffers still frozen for clean evaluation, and a looser clip so updates are visible without becoming unstable.

It also saves simple baseline-specific diagnostics that are easy to explain in a dissertation. For GA, these are the mean forget-set cross-entropy and gradient norm at each unlearning step. For SCRUB, they are the student-teacher KL divergence on the forget and retain sets, the retain-set cross-entropy, and retain/forget accuracy. Together these show whether SCRUB is separating from the teacher on forgotten data while still staying close on retained data.

In [ ]:
import warnings

from IPython.display import Image as IPyImage, display

from gradient_ascent.notebook_helpers import (
    ALGORITHM_ORDER,
    prepare_notebook_runtime,
    run_notebook_core_experiment,
    save_ga_diagnostics,
    save_scrub_diagnostics,
)

try:
    import wandb
except ImportError:
    wandb = None

warnings.filterwarnings("ignore", category=DeprecationWarning)

NUM_CLASSES = 10
OUT_DIR = "out"
resnet_model_depth = 50

runtime = prepare_notebook_runtime(
    num_classes=NUM_CLASSES,
    out_dir=OUT_DIR,
    model_depth=resnet_model_depth,
)

device = runtime.device
USE_BF16 = runtime.use_bf16
trainset = runtime.trainset
testset = runtime.testset
use_cuda = runtime.use_cuda
num_workers = runtime.num_workers
model_factory = runtime.model_factory
core_config = runtime.core_config

core_artifacts, wandb_run = run_notebook_core_experiment(runtime, wandb_module=wandb)

display(IPyImage(filename=core_artifacts.original_vs_retrain_plot_path))
for algorithm_key in ALGORITHM_ORDER:
    artifact = core_artifacts.algorithm_artifacts[algorithm_key]
    display(IPyImage(filename=artifact.classwise_percent_plot_path))
    display(IPyImage(filename=artifact.classwise_absolute_plot_path))

for key, value in core_artifacts.summary_metrics.items():
    print(f"{key}: {value:.3f}")

ga_csv_path, ga_plot_path = save_ga_diagnostics(runtime, core_artifacts, wandb_run=wandb_run, wandb_module=wandb)
print(f"Saved GA diagnostics CSV to {ga_csv_path}")
print(f"Saved GA diagnostics plot to {ga_plot_path}")
display(IPyImage(filename=ga_plot_path))

scrub_csv_path, scrub_plot_path = save_scrub_diagnostics(
    runtime,
    core_artifacts,
    wandb_run=wandb_run,
    wandb_module=wandb,
)
print(f"Saved SCRUB diagnostics CSV to {scrub_csv_path}")
print(f"Saved SCRUB diagnostics plot to {scrub_plot_path}")
display(IPyImage(filename=scrub_plot_path))


## 5. Shared similarity helpers

Defines reusable activation extraction, metric evaluation, and plotting helpers used by downstream trajectory analysis. Standalone final pairwise snapshot generation is removed to avoid redundancy.

In [ ]:
# Shared similarity setup is imported from the reusable package.
from gradient_ascent.notebook_helpers import prepare_similarity_setup

similarity_setup = prepare_similarity_setup()
layer_names = similarity_setup.layer_names
metrics = similarity_setup.metrics
higher_better_metrics = similarity_setup.higher_better_metrics
lower_better_metrics = similarity_setup.lower_better_metrics
plot_metric_names = similarity_setup.plot_metric_names

print("Shared similarity helpers and metrics loaded from gradient_ascent.notebook_helpers.")

## 6. Epoch-wise unlearning trajectories, MIA, and animations

Runs snapshot trajectories for all five baselines (GA, SSD, SalUn, certified removal, and SCRUB), computes MIA trajectories, and generates evolving similarity summaries and GIFs for both references: unlearned-vs-retrained and unlearned-vs-original.

In [ ]:
# Unlearning algorithm comparison trajectories across all five baselines.
# Computes artifact files in the package, then displays them compactly here.

from IPython.display import Image as IPyImage, display

from gradient_ascent.notebook_helpers import run_notebook_trajectory_experiment

if "similarity_setup" not in globals():
    raise RuntimeError("Run the shared similarity helper cell before this cell.")

trajectory_artifacts, trajectory_wandb_run = run_notebook_trajectory_experiment(
    runtime,
    core_artifacts,
    similarity_setup,
    wandb_module=wandb,
)

for algorithm_key in ALGORITHM_ORDER:
    mia_artifact = trajectory_artifacts.mia_artifacts[algorithm_key]
    display(IPyImage(filename=mia_artifact.grid_plot_path))
    display(IPyImage(filename=mia_artifact.control_plot_path))
    for reference_key in ["retrained", "original"]:
        similarity_artifact = trajectory_artifacts.similarity_artifacts[algorithm_key][reference_key]
        display(IPyImage(filename=similarity_artifact.summary_plot_path))
        display(IPyImage(filename=similarity_artifact.gif_path))


## 7. Combined cross-algorithm trajectory comparison

Overlays all five baselines, including SCRUB, in a single consolidated similarity + MIA comparison figure.

In [ ]:
# Integrated combined comparison figure: similarity + MIA trajectories.
# Reads saved trajectory CSV artifacts and displays the final summary inline.

from IPython.display import Image as IPyImage, display

from gradient_ascent.notebook_helpers import save_notebook_combined_comparison

combined_path = save_notebook_combined_comparison(runtime, wandb_module=wandb)
print(f"Saved integrated comparison figure to {combined_path}")
display(IPyImage(filename=combined_path))
